# PHYS 690 - Lecture 15: Monte Carlo Sampling and Integration

## One damped Ramsey experiment, many random samples

Unit 4 will follow one model across five lectures: a two-level quantum system measured after a variable delay. Its damped oscillation is deliberately close to the quantum-optics model used in Lecture 6.  In a Ramsey-type measurement, repeated preparation and measurement produce binary outcomes. The excited-state probability is modeled as

$$
p_e(t \mid \boldsymbol{\theta}) = \frac{1}{2}\left[1 + C e^{-t/\tau}\cos(\omega t + \phi)\right],
$$

where $\boldsymbol{\theta} = (C, \tau, \omega, \phi)$ contains the contrast, coherence time, angular frequency, and phase. This model appears in qubit calibration and quantum sensing; see the [Qiskit Experiments RamseyXY documentation](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.library.characterization.RamseyXY.html) and the open review [Degen, Reinhard, and Cappellaro, *Quantum sensing*](https://arxiv.org/abs/1611.02427).

## Learning goals and class plan

By the end of class, you should be able to:

1. explain what a pseudorandom-number generator and seed do;
2. express an integral as an expectation value;
3. explain when Monte Carlo integration is useful as the number of variables grows;
4. construct Monte Carlo estimators and their standard errors;
5. verify the characteristic $1/\sqrt{N}$ sampling-error scaling; and
6. use stratified sampling to reduce variance without changing the target integral.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.stats import norm

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

## Why use Monte Carlo integration in physics?

Many useful physics quantities are averages over several variables. Examples include averages over position and momentum in phase space, averages over uncertain experimental conditions, and integrating a detector response over the particles that can reach the apparatus.

For one or two variables, a regular grid and numerical quadrature are often simple and efficient. But if we use $m$ grid points along each of $d$ dimensions, the grid contains $m^d$ evaluations. That cost grows rapidly as more variables are added. Monte Carlo integration instead draws representative points from the relevant ranges or distributions and averages the function values. Its characteristic error decreases like $1/\sqrt{N}$, where $N$ is the number of random samples; the rate does not acquire an additional power of $N$ just because the number of dimensions increases.

Monte Carlo is not automatically more accurate or faster in every problem. The variance of the sampled values still matters, and low-dimensional quadrature is usually preferable when it is available. We will begin with an easy one-dimensional Ramsey example to understand the estimator, then extend the same probability model to a two-dimensional average over delay and detuning.


## Part 1: A reproducible stream of pseudorandom numbers

A computer normally produces **pseudorandom** values: a deterministic algorithm maps an internal state to a sequence that passes useful statistical tests. A seed initializes that state. Reusing a seed reproduces a sequence, which is valuable for debugging and teaching, but it does not make the outcomes physically deterministic.

Use NumPy's `Generator` interface rather than the older global random functions. One generator should usually be created near the beginning of a calculation and passed explicitly when needed.

**Discuss:** When is a fixed seed scientifically helpful? When might reporting only one seed hide a fragile result?

In [ ]:
# Create two independent random number generators with the same seed.
rng_a = np.random.default_rng(690)
rng_b = np.random.default_rng(690)

# Identical seeds and identical calls reproduce the same sequence.
reproducibility_df = pd.DataFrame({
    "draw_from_a": rng_a.random(6),
    "draw_from_b": rng_b.random(6),
})
reproducibility_df["difference"] = (
    reproducibility_df["draw_from_a"] - reproducibility_df["draw_from_b"]
)
reproducibility_df

### The repeated physical model

We use normalized time in microseconds and angular frequency in $\mathrm{rad}/\mu\mathrm{s}$. The probability remains between zero and one because $0 \leq C \leq 1$. These parameter values will be reused throughout Unit 4.

In [ ]:
# Define the shared Ramsey probability model.
def ramsey_probability(t, contrast, tau, omega, phase):
    envelope = np.exp(-np.asarray(t) / tau)
    oscillation = np.cos(omega * np.asarray(t) + phase)
    return 0.5 * (1.0 + contrast * envelope * oscillation)


true_parameters = {
    "contrast": 0.78,
    "tau": 2.40,              # microseconds
    "omega": 2.0 * np.pi * 0.85,  # rad/microsecond
    "phase": 0.25,            # rad
}

time_plot = np.linspace(0.0, 5.0, 600)
model_df = pd.DataFrame({
    "time_us": time_plot,
    "excited_probability": ramsey_probability(time_plot, **true_parameters),
})

fig, ax = plt.subplots()
ax.plot(model_df["time_us"], model_df["excited_probability"], lw=2)
ax.set(xlabel=r"Delay $t$ ($\mu$s)", ylabel=r"$p_e(t)$",
       title="Damped Ramsey probability")
ax.set_ylim(0.0, 1.0)
plt.show()

## Part 2: Emulating a Ramsey scan with Monte Carlo sampling

### What would the experiment do?

In a Ramsey experiment, each trial prepares the same two-level system, lets it evolve for a chosen delay $t$, and measures whether it is in the excited state. The actual readout from one trial is binary:

$$
X =
\begin{cases}
1, & \text{excited state is measured},\\
0, & \text{ground state is measured}.
\end{cases}
$$

At fixed delay $t$, the probability of measuring $X=1$ is the model value $p_e(t)$. Repeating the preparation and measurement many times at that delay estimates $p_e(t)$ from the fraction of excited-state outcomes.

### What quantity will we estimate?

Imagine choosing the delay uniformly at random from $a$ to $b$. The quantity of interest here is the **area under the model probability curve**,

$$
I = \int_a^b p_e(t)\,dt.
$$

Equivalently, $I/(b-a)$ is the average excited-state probability across that delay window. We can estimate this integral by drawing delays $T_i$ uniformly from $[a,b]$, evaluating the model probability at each delay, and averaging:

$$
\widehat{I}_N = (b-a)\frac{1}{N}\sum_{i=1}^{N}p_e(T_i).
$$

Here $N$ is the number of randomly selected delays, and $\widehat{I}_N$ is the Monte Carlo estimator: a random approximation to the integral. In this first calculation, we use the model probability $p_e(T_i)$ at each sampled delay rather than simulating a binary detector click. This isolates the error from sampling delays. Lecture 16 will add the binary shot outcomes and their additional variability.

### How uncertain is the estimate?

Let $s_p$ be the sample standard deviation of the $N$ sampled probability values $p_e(T_i)$:

$$
s_p = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N}\left[p_e(T_i)-\overline{p}_e\right]^2},
\qquad
\overline{p}_e = \frac{1}{N}\sum_{i=1}^{N}p_e(T_i).
$$

The **standard error** estimates how much the integral estimate would typically vary if we repeated the random-delay calculation. For this estimator,

$$
\mathrm{SE}\!\left(\widehat{I}_N\right) = (b-a)\frac{s_p}{\sqrt{N}}.
$$

This standard error is Monte Carlo sampling uncertainty. It is not uncertainty in the physical parameters, and it does not yet include the randomness of binary detector outcomes.


### Why the estimator works

If the delays are drawn uniformly from $[a,b]$, the average of the sampled values $p_e(T_i)$ approaches the average value of the probability curve over that interval. Multiplying that sample average by the interval width gives the area $I$.

The estimator is **unbiased**: over many independent repetitions of the random-delay calculation, its average equals the target integral. A single estimate will generally differ from the integral. Its typical variation is described by the standard error above, which decreases in proportion to $1/\sqrt{N}$ when the sampled values are independent and have finite variance.

For a sufficiently large sample, the estimate's deviations from the target are often approximately described by a normal distribution. This approximation can be poor for small samples, rare-event problems, or cases where the sampled values have very large variation. Repeating the calculation with independent random seeds provides a useful check.


In [ ]:
# Define the finite delay window and a scalar integrand for scipy.integrate.quad.
t_min, t_max = 0.0, 5.0

def probability_at_time(t):
    return ramsey_probability(t, **true_parameters)

# Use adaptive quadrature as a high-accuracy benchmark for this one-dimensional case.
reference_integral, quadrature_error = quad(probability_at_time, t_min, t_max)
reference_average = reference_integral / (t_max - t_min)

print(f"Reference integral = {reference_integral:.8f}")
print(f"Average excitation probability = {reference_average:.8f}")
print(f"quad error estimate = {quadrature_error:.2e}")

### Observe the error scaling

Monte Carlo is not the best algorithm for this smooth one-dimensional integral; quadrature is faster. Its strength is that the same estimator generalizes to high-dimensional integrals and complicated sampling regions. Here the easy benchmark lets us inspect the method honestly.

In [ ]:
def direct_mc_integral(n_samples, rng):
    # Draw delays from the uniform density used in the expectation.
    sampled_t = rng.uniform(t_min, t_max, size=n_samples)
    sampled_values = probability_at_time(sampled_t)

    # Scale the sample mean and its standard error by the interval width.
    width = t_max - t_min
    estimate = width * np.mean(sampled_values)
    standard_error = width * np.std(sampled_values, ddof=1) / np.sqrt(n_samples)
    return estimate, standard_error


# Repeat the estimate at increasing sample sizes with one reproducible stream.
scaling_rng = np.random.default_rng(1501)
sample_sizes = np.array([100, 300, 1000, 3000, 10000, 30000])
scaling_rows = []

for n_samples in sample_sizes:
    estimate, standard_error = direct_mc_integral(n_samples, scaling_rng)
    scaling_rows.append({
        "n_samples": n_samples,
        "estimate": estimate,
        "absolute_error": abs(estimate - reference_integral),
        "estimated_standard_error": standard_error,
    })

scaling_df = pd.DataFrame(scaling_rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].errorbar(scaling_df["n_samples"], scaling_df["estimate"],
                 yerr=scaling_df["estimated_standard_error"], fmt="o-")
axes[0].axhline(reference_integral, color="black", ls="--", label="quadrature")
axes[0].set(xscale="log", xlabel="Number of samples", ylabel="Integral estimate")
axes[0].legend()

axes[1].loglog(scaling_df["n_samples"],
               scaling_df["estimated_standard_error"], "o-", label="estimated SE")
reference_line = scaling_df["estimated_standard_error"].iloc[0] * np.sqrt(
    scaling_df["n_samples"].iloc[0] / scaling_df["n_samples"]
)
axes[1].loglog(scaling_df["n_samples"], reference_line, "--", label=r"$1/\sqrt{N}$")
axes[1].set(xlabel="Number of samples", ylabel="Estimated standard error")
axes[1].legend()
fig.suptitle("Direct Monte Carlo integration")
fig.tight_layout()
plt.show()

## Part 3: In-class coding activity 1 - Construct and report an estimate

Estimate the integral with $N = 4000$ independent uniform samples. Report the estimate, its standard error, and the approximate 1-sigma interval $\widehat{I}_N \pm \mathrm{SE}(\widehat{I}_N)$. Then decide whether the quadrature benchmark is consistent with that interval.

When the estimate's sampling distribution is approximately Gaussian, about 68.3% of repeated estimates fall within one standard error of the true integral. This is the central 68% interval under that Gaussian approximation.

Before coding, identify which line represents randomness and which line represents the deterministic physical model.

In [ ]:
activity_rng = np.random.default_rng(1515)
activity_n = 4000
activity_width = t_max - t_min

# TODO: Draw activity_n uniform delays over the integration interval.
activity_t = ...
# TODO: Evaluate the Ramsey probability at every sampled delay.
activity_values = ...
# TODO: Calculate the Monte Carlo integral estimate.
activity_estimate = ...
# TODO: Estimate the standard error using the sample standard deviation.
activity_standard_error = ...
# TODO: Form the lower and upper limits of an approximate 1-sigma interval.
activity_interval = ...

activity_summary_df = pd.DataFrame({
    "quantity": ["estimate", "standard error", "1-sigma lower", "1-sigma upper", "benchmark"],
    "value": [activity_estimate, activity_standard_error,
              activity_interval[0], activity_interval[1], reference_integral],
})
activity_summary_df

## Part 4: A two-dimensional Ramsey average

The one-dimensional example averages over delay. A real experiment can also have a spread in detuning: different trials may experience a small offset $\Delta$ from the nominal angular frequency $\omega$. For this worked example, suppose both the delay and detuning are sampled uniformly over finite ranges. The detuning range is a simple model of a controlled calibration window.

For a trial with delay $t$ and detuning $\Delta$, use

$$
p_e(t,\Delta)=\frac{1}{2}\left[1+C e^{-t/\tau}\cos\!\left((\omega+\Delta)t+\phi\right)\right].
$$

The quantity we want is the average predicted excited-state probability across the rectangle $a\leq t\leq b$ and $-\Delta_{\max}\leq\Delta\leq\Delta_{\max}$:

$$
\overline{p}_e=
\frac{1}{(b-a)(2\Delta_{\max})}
\int_a^b\!dt\int_{-\Delta_{\max}}^{\Delta_{\max}}\!d\Delta\;p_e(t,\Delta).
$$

Each Monte Carlo sample is now a **pair** $(t_i,\Delta_i)$. The estimator is the average probability across the sampled pairs,

$$
\widehat{\overline{p}}_{e,N}=\frac{1}{N}\sum_{i=1}^{N}p_e(t_i,\Delta_i).
$$

If $s_p$ is the sample standard deviation of these $N$ probability values, the estimated standard error on this average is $s_p/\sqrt{N}$. Multiplying the average and its standard error by the rectangle area gives the corresponding integral estimate and its standard error.

The binary measurement outcome is still not simulated here; this calculation estimates an average probability. Lecture 16 will separately model binary shot outcomes and correlated calibration uncertainty.

In two dimensions, a midpoint grid is still feasible and serves as a useful reference. The Monte Carlo example demonstrates how to sample jointly and estimate uncertainty; it does not claim that random sampling is the fastest method for this small problem.


In [ ]:
# Set the finite time and detuning ranges for the 2D average.
detuning_half_width = 0.35  # rad/microsecond
time_width = t_max - t_min
detuning_width = 2.0 * detuning_half_width
rectangle_area = time_width * detuning_width


# Evaluate the probability on a midpoint grid as a deterministic reference.
n_time_grid = 160
n_detuning_grid = 140
time_midpoints = t_min + (np.arange(n_time_grid) + 0.5) * time_width / n_time_grid
detuning_midpoints = (
    -detuning_half_width
    + (np.arange(n_detuning_grid) + 0.5) * detuning_width / n_detuning_grid
)
time_mesh, detuning_mesh = np.meshgrid(time_midpoints, detuning_midpoints)

grid_probabilities = 0.5 * (
    1.0
    + true_parameters["contrast"]
    * np.exp(-time_mesh / true_parameters["tau"])
    * np.cos(
        (true_parameters["omega"] + detuning_mesh) * time_mesh
        + true_parameters["phase"]
    )
)
grid_average_probability = np.mean(grid_probabilities)
grid_integral = rectangle_area * grid_average_probability

# Store the grid in a DataFrame so its coordinates and values stay labeled.
probability_surface_df = pd.DataFrame({
    "time_us": time_mesh.ravel(),
    "detuning_rad_us": detuning_mesh.ravel(),
    "excited_probability": grid_probabilities.ravel(),
})
surface_pivot_df = probability_surface_df.pivot(
    index="detuning_rad_us", columns="time_us", values="excited_probability"
)


# Draw independent delay-detuning pairs and evaluate the same model.
two_d_rng = np.random.default_rng(1520)
two_d_n = 12000
two_d_samples_df = pd.DataFrame({
    "time_us": two_d_rng.uniform(t_min, t_max, size=two_d_n),
    "detuning_rad_us": two_d_rng.uniform(
        -detuning_half_width, detuning_half_width, size=two_d_n
    ),
})
two_d_samples_df["excited_probability"] = 0.5 * (
    1.0
    + true_parameters["contrast"]
    * np.exp(-two_d_samples_df["time_us"] / true_parameters["tau"])
    * np.cos(
        (true_parameters["omega"] + two_d_samples_df["detuning_rad_us"])
        * two_d_samples_df["time_us"]
        + true_parameters["phase"]
    )
)

two_d_average = two_d_samples_df["excited_probability"].mean()
two_d_average_se = (
    two_d_samples_df["excited_probability"].std(ddof=1) / np.sqrt(two_d_n)
)
two_d_integral = rectangle_area * two_d_average
two_d_integral_se = rectangle_area * two_d_average_se

two_d_summary_df = pd.DataFrame({
    "method": ["midpoint grid", "Monte Carlo"],
    "average_probability": [grid_average_probability, two_d_average],
    "integral_over_rectangle": [grid_integral, two_d_integral],
    "integral_standard_error": [np.nan, two_d_integral_se],
})
display(two_d_summary_df)

# Show the probability surface and the sampled delay-detuning pairs.
fig, ax = plt.subplots(figsize=(8.5, 4.8))
image = ax.pcolormesh(
    surface_pivot_df.columns.to_numpy(),
    surface_pivot_df.index.to_numpy(),
    surface_pivot_df.to_numpy(),
    shading="auto",
    cmap="viridis",
)
sampled_points_df = two_d_samples_df.iloc[:300]
ax.scatter(sampled_points_df["time_us"], sampled_points_df["detuning_rad_us"],
           s=9, facecolors="none", edgecolors="white", linewidths=0.5,
           label="first 300 Monte Carlo samples")
ax.set_xlabel(r"Delay $t$ ($\mu$s)")
ax.set_ylabel(r"Detuning offset $\Delta$ (rad/$\mu$s)")
ax.set_title(r"Ramsey probability across delay and detuning")
fig.colorbar(image, ax=ax, label=r"$p_e(t,\Delta)$")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

print(f"Grid average probability = {grid_average_probability:.6f}")
print(f"Monte Carlo average probability = {two_d_average:.6f} +/- {two_d_average_se:.6f}")
print(f"Grid integral = {grid_integral:.6f}")
print(f"Monte Carlo integral = {two_d_integral:.6f} +/- {two_d_integral_se:.6f}")


### How the same idea extends to more dimensions

For $d$ variables, one Monte Carlo sample is a point with $d$ coordinates, and the estimator is still the average of the integrand evaluated at sampled points. A regular grid with $m$ points along each coordinate requires $m^d$ evaluations, while Monte Carlo uses the chosen number $N$ of joint samples. Its standard error still scales as $1/\sqrt{N}$, although the variance of the sampled integrand can become large in a difficult high-dimensional problem. Sampling from physically relevant distributions and using variance-reduction methods can help.

Examples where the integral may involve many variables include:

- **Plasma physics:** average collision or reaction rates over particle velocities, collision angles, and local plasma conditions; integrate a distribution function over position and velocity space.
- **Quantum optics:** average photon-count probabilities over fluctuating laser intensity, detuning, phase, polarization, and detector timing jitter.
- **Quantum materials:** integrate band energies or occupation probabilities across a three-dimensional Brillouin zone, or average observables over thermal and disorder configurations.
- **Nuclear and particle physics:** integrate a differential cross section over multidimensional final-state phase space, or average detector response over particle momenta, production positions, and detector variables.

The 2D Ramsey calculation is small enough that midpoint quadrature remains practical. These higher-dimensional examples show why the same sampling-and-average strategy becomes valuable when a grid would require too many evaluations.


## Part 5: Variance reduction by stratified sampling

Direct sampling can leave accidental gaps and clusters. **Stratified sampling** divides the interval into $N$ equal pieces and draws one random point from each. The estimator remains an average over the full interval, but the sample covers the domain more evenly.

For stratum $i$,

$$
T_i = a + (b-a)\frac{i+U_i}{N}, \qquad U_i \sim \mathrm{Uniform}(0,1).
$$

This does not guarantee lower variance for every integrand, but it is effective for many smooth functions. It is also a bridge to quasi-Monte Carlo methods, which construct deliberately space-filling point sets.

### In-class coding activity 2 - Compare two sampling designs

Complete the stratified estimator. The following completed cell will repeat both estimators 250 times at the same sample size, so the spread across repetitions measures their actual sampling variability.

In [ ]:
def stratified_mc_integral(n_samples, rng):
    # TODO: Draw one uniform offset inside each of n_samples equal strata.
    unit_interval_samples = ...
    # TODO: Map the unit-interval samples to [t_min, t_max].
    sampled_t = ...
    # TODO: Return the interval width times the mean model value.
    estimate = ...
    return estimate

In [ ]:
# This comparison assumes the activity function above is complete.
comparison_rng = np.random.default_rng(1516)
comparison_n = 256
n_repetitions = 250
comparison_rows = []

for repetition in range(n_repetitions):
    direct_estimate, _ = direct_mc_integral(comparison_n, comparison_rng)
    stratified_estimate = stratified_mc_integral(comparison_n, comparison_rng)
    comparison_rows.extend([
        {"method": "independent uniform", "estimate": direct_estimate},
        {"method": "stratified", "estimate": stratified_estimate},
    ])

comparison_df = pd.DataFrame(comparison_rows)
method_summary_df = comparison_df.groupby("method")["estimate"].agg(
    mean="mean", empirical_standard_deviation="std"
).reset_index()
display(method_summary_df)

fig, ax = plt.subplots()
for method, group in comparison_df.groupby("method"):
    ax.hist(group["estimate"], bins=24, alpha=0.55, density=True, label=method)
ax.axvline(reference_integral, color="black", ls="--", label="benchmark")
ax.set(xlabel="Integral estimate", ylabel="Density",
       title=f"Repeated estimates with N={comparison_n}")
ax.legend()
plt.show()

## Part 6: What Monte Carlo uncertainty means

Three distinctions will matter throughout Unit 4:

| Source of variation | Question answered | Reduced by more Monte Carlo draws? |
|---|---|---|
| Monte Carlo sampling error | How accurately did the algorithm estimate an expectation? | Yes |
| Experimental shot noise | How much would repeated physical data sets vary? | Only by collecting more physical shots |
| Parameter or calibration uncertainty | Which model inputs remain uncertain? | No; it must be measured or inferred |

A large number of simulated samples can make the first row tiny while leaving the other two unchanged. Reporting only the number of Monte Carlo draws is therefore not a complete uncertainty statement.

**Exit question:** If the Monte Carlo standard error is halved, by what factor did the sample count increase?

### Choosing a sampling strategy

| Method | Main idea | Strength | Limitation |
|---|---|---|---|
| Direct Monte Carlo | Draw from the density defining the expectation | Simple and dimension-independent $1/\sqrt{N}$ scaling | Can waste samples where the integrand contributes little |
| Stratified sampling | Force coverage of subregions | Often reduces variance for smooth functions | More bookkeeping in many dimensions |
| Importance sampling | Draw from a proposal concentrated in important regions and reweight | Powerful for localized integrands and rare events | Poor proposals produce extreme weights |
| Quasi-Monte Carlo | Use low-discrepancy deterministic points | Often converges faster for smooth moderate-dimensional problems | Error estimation and high-dimensional performance require care |

No method removes the need for a convergence study. Increase the sample count, repeat with independent randomizations, and verify that the observed spread is compatible with the reported Monte Carlo error.

## References and further reading

- [Unit 4 course resources](../resources/unit-4-monte-carlo-inference.md)
- [NumPy random `Generator` documentation](https://numpy.org/doc/stable/reference/random/generator.html)
- [SciPy quasi-Monte Carlo documentation](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html)
- [Allen Downey, *Think Bayes 2*](https://allendowney.github.io/ThinkBayes2/)
- [Mark Newman, *Computational Physics* resources](https://websites.umich.edu/~mejn/cp2/)